# Macro Distance Model for Factor Timing

Replication and extension of [Man Group's remime-based similarity model]

## Methodology

1. **7 financial state variables** (oil, copper, vix, rate)
2. **Normalise** each variable via rolling z-score (10-year window), winsorised at +/-3
3. **Euclidean distance** between current month and all historical months
4. **Exclude last 36 months** from similarity calculation to avioid momentum loading
5. **Forecast factor returns** using similarity-weighted historical averages
6. **Backtest** against Fama-French 6 factors + Momentum

### Man group benchmark
- Q1-Q5 spread portfolio sharpe: **0.82**
- Correlation to static long-only: **0.37**

In [ ]:
# Imports
import pandas as pd
import numpy as np
from datetime import datetime
import yfinance as yf
import requests
import zipfile
import io
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import plotly.io as pio
pio.renderers.default = 'notebook'

pd.set_option('display.max_columns', None)
pd.set_option('display.max_row', 100)
pd.set_option('display.float_format', '{:.4f}'.format)

COLOURS = sns.color_palette('muted').as_hex()

## 1. Load State Variables

Source the 7 financial indicators from public data. All variables are converted to 12-month changes.

| Oil price | Yahoo Finance | `BZ=F` |
| Copper price | Yahoo Finance | `HG=F` |


In [2]:
# ==========================================================================
# Configuration
# ==========================================================================

# Distance model parameters
DISTANCE_CONFIG = dict(
    norm_window=120,
    winsorize_limit=3.0,
    exclusion_window=36,
    min_periods=24
)

DATA_START = '1990-01-01'

print("Configuration loaded")
print(f"Distance config:  {DISTANCE_CONFIG}")

Configuration loaded
Distance config:  {'norm_window': 120, 'winsorize_limit': 3.0, 'exclusion_window': 36, 'min_periods': 24}


In [3]:
# ==========================================================================
# Load state variables from sources
# ==========================================================================

yf_tickers = {
    'oil': 'BZ=F',
    'copper': 'HG=F',
    'vix': '^VIX',
    'us_10y': 'TNX'
}

print('Downloading from Yahoo Finance data...')
yf_data = {}
for name, ticker in yf_tickers.items():
    try:
        df = yf.download(ticker, start=DATA_START, progress=False, auto_adjust=True)
        if isinstance(df.columns, pd.MultiIndex):
            series = df['Close'].iloc[:, 0]
        else:
            series = df['Close']
        series.index = pd.to_datetime(series.index)
        yf_data[name] = series.dropna()
    except Exception as e:
        print(f'  {name} ({ticker}): FAILED - {e}')

fred_series = {
    'credit_spread': 'BAMLC0A4CBBB',    # ICE BofA BBB US Corporate Index
    'cpi': 'CPIAUCSL'                   # CPI for all urban consumers
}

print('\nDownloading FRED data...')
for name, series_id in fred_series.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd=1990-01-01'
    try:
        df = pd.read_csv(url, parse_dates=['observation_date'], index_col='observation_date')
        df.columns = ['value']
        df['value'] = pd.to_numeric(df['value'], errors='coerce')
        yf_data[name] = df['value'].dropna()
    except Exception as e:
        print(f'  {name} ({series_id}: FAILED - {e})')

print(f'\nLoaded {len(yf_data)} series: {list(yf_data.keys())}')



Loaded 6 series: ['oil', 'copper', 'vix', 'us_10y', 'credit_spread', 'cpi']


In [4]:
yf_data['us_10y']

Date
2019-04-22   1.2300
2019-04-23   1.1900
2019-04-24   1.1900
2019-04-25   1.1500
2019-04-26   1.1500
Name: TNX, dtype: float64

In [5]:
# ==========================================================================
# Transform state variable
# ==========================================================================

# Resample to month end
macro_monthly = pd.DataFrame()
for name, series in yf_data.items():
    monthly = series.resample('ME').last()
    macro_monthly[name] = monthly

state_vars = pd.DataFrame(index=macro_monthly.index)

# Price like variable as % change
for var in ['oil', 'copper', 'vix']:
    if var in macro_monthly.columns:
        state_vars[var] = macro_monthly[var].pct_change(12)

# Rate / Yield as absolute change
for var in ['credit_spread', 'us_10y']:
    if var in macro_monthly.columns:
        state_vars[var] = macro_monthly[var].diff(12)

# Equity dispersion proxy: cross-sectional vol of FF industry return
# We use the Fama-French 10 industry portfolio return to measure dispersion
print('\nLoading equity dispersion proxy...')
try:
    disp_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/10_Industry_Portfolios_CSV.zip'
    resp = requests.get(disp_url, timeout=30)
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        fname = zf.namelist()[0]
        with zf.open(fname) as f:
            lines = f.read().decode('utf-8').splitlines()

    # Parse monthly returns section
    header_idx = None
    data_lines = []
    for i, line in enumerate(lines):
        if 'NoDur' in line and 'Durbl' in line:
            header_idx = i
            continue
        if header_idx is not None:
            stripped = line.strip()
            if stripped == '' or 'Annual' in stripped:
                break
            parts = stripped.split(',')
            if len(parts) >= 2 and len(parts[0].strip()) == 6:
                data_lines.append(stripped)
    
    if data_lines:
        from io import StringIO
        header_line = lines[header_idx].strip()
        csv_text = header_line + '\n' + '\n'.join(data_lines)
        ind_df = pd.read_csv(StringIO(csv_text))
        ind_df.rename(columns={ind_df.columns[0]: 'date'}, inplace=True)
        ind_df['date'] = pd.to_datetime(ind_df['date'].astype(str).str.strip(), format='%Y%m')
        ind_df['date'] = ind_df['date'] + pd.offsets.MonthEnd(0)
        ind_df = ind_df.set_index('date')
        ind_df = ind_df.apply(pd.to_numeric, errors='coerce') / 100

        # Cross sectional standard deviation = dispersion measure
        dispersion = ind_df.std(axis=1)
        dispersion.name = 'dispersion'
        state_vars['dispersion'] = dispersion.diff(12)  # 12M absolute change
        print(f'   Dispersion proxy: {dispersion.dropna().shape[0]} months')
    else:
        print('   Could not parse industry portfolios')
except Exception as e:
    print(f'   Dispersion load failed: {e}')

# Real rates = 10Y yield minus YoY CPI inflation
if 'us_10y' in macro_monthly.columns and 'cpi' in macro_monthly.columns:
    cpi_yoy = macro_monthly['cpi'].pct_change(12) * 100
    real_rate = macro_monthly['us_10y'] - cpi_yoy
    state_vars['real_rates'] = real_rate.diff(12)
    print(f'Real rate derived: {state_vars['real_rates'].dropna().shape[0]} months')

state_vars = state_vars.dropna(how='all')
print(f'Date range: {state_vars.index.min().date()} to {state_vars.index.max().date()}')
print('Summary:')
for col in state_vars.columns:
    s = state_vars[col].dropna()
    print(f'  {col:15s}: n={len(s):4d}, mean={s.mean():8.4f}, std={s.std():8.4f}, min={s.min():8.4f}, max={s.max():8.4f}')

# print(f'\nMissing data (%): {state_vars.isna().mean() * 100:.1f}')


Loading equity dispersion proxy...
   Dispersion proxy: 1196 months
Real rate derived: 0 months
Date range: 2007-07-31 to 2026-04-30
Summary:
  oil            : n= 214, mean=  0.0520, std=  0.3897, min= -0.6675, max=  1.7942
  copper         : n= 214, mean=  0.0595, std=  0.2980, min= -0.6038, max=  1.3853
  vix            : n= 214, mean=  0.0856, std=  0.5139, min= -0.6377, max=  2.9052
  credit_spread  : n= 214, mean= -0.0815, std=  1.3601, min= -5.3600, max=  5.3900
  us_10y         : n=   0, mean=     nan, std=     nan, min=     nan, max=     nan
  dispersion     : n= 224, mean=  0.0012, std=  0.0260, min= -0.1046, max=  0.0947
  real_rates     : n=   0, mean=     nan, std=     nan, min=     nan, max=     nan


TypeError: unsupported format string passed to Series.__format__

## 2. Load Factor Returns

Load Fama-French 5 factors + Momentum from the Kenneth French Data Library

In [6]:
# ==========================================================================
# Load factor returns
# ==========================================================================

def load_french_csv(url, skip_footer_sections=True) -> pd.DataFrame:
    """
    Download and parse factor csv
    """
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(resp.content)) as zf:
        fname = zf.namelist()[0]
        with zf.open(fname) as f:
            lines = f.read().decode('utf-8').splitlines()

    # Find header row (contains 'Mkt-Rf')
    header_idx = None
    for i, line in enumerate(lines):
        if 'Mkt-RF' in line or ',Mom' in line or 'WML' in line:
            header_idx = i
            break

    if header_idx is None:
        raise ValueError('Could not find header row')
    
    # Parse monthly data row
    data_lines = []
    for line in lines[header_idx + 1:]:
        stripped = line.strip()
        if stripped == '' or not stripped[0].isdigit():
            break
        parts = stripped.split(',')
        if len(parts) >= 2 and len(parts[0].strip()) == 6:
            data_lines.append(stripped)

    from io import StringIO
    header_line = lines[header_idx].strip()
    csv_text = header_line + '\n' + '\n'.join(data_lines)
    ind_df = pd.read_csv(StringIO(csv_text))
    ind_df.rename(columns={ind_df.columns[0]: 'date'}, inplace=True)
    ind_df['date'] = pd.to_datetime(ind_df['date'].astype(str).str.strip(), format='%Y%m')
    ind_df['date'] = ind_df['date'] + pd.offsets.MonthEnd(0)
    ind_df = ind_df.set_index('date')
    ind_df = ind_df.apply(pd.to_numeric, errors='coerce') / 100
    return ind_df
    
# Load FF5 factors
print('Loading Fama-French 5 factors...')
ff5_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_CSV.zip'
ff5 = load_french_csv(ff5_url)

# Load momentum
print('Loading Momentum factor...')
mom_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_CSV.zip'
mom = load_french_csv(mom_url)

# Combine
factor_returns = ff5[['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']].copy()
factor_returns.columns = ['Market', 'Size', 'Value', 'Quality', 'Investment']
factor_returns['Momentum'] = mom.iloc[:, 0]

factor_returns = factor_returns.dropna()

print(f'\nFactor returns: {factor_returns.shape}')
print(f'Date range: {factor_returns.index.min().date} to {factor_returns.index.max().date}')
print(f'Factors: {list(factor_returns.columns)}')
print(f'\nAnnualized Sharpe ratios:')
for col in factor_returns.columns:
    s = factor_returns[col]
    sharpe = (s.mean() / s.std()) * np.sqrt(12)
    print(f'  {col:12s}: {sharpe:.2f}')

Loading Fama-French 5 factors...
Loading Momentum factor...

Factor returns: (752, 6)
Date range: <bound method Timestamp.date of Timestamp('1963-07-31 00:00:00')> to <bound method Timestamp.date of Timestamp('2026-02-28 00:00:00')>
Factors: ['Market', 'Size', 'Value', 'Quality', 'Investment', 'Momentum']

Annualized Sharpe ratios:
  Market      : 0.46
  Size        : 0.21
  Value       : 0.34
  Quality     : 0.42
  Investment  : 0.42
  Momentum    : 0.50


## 3. Fit Distance Model

Implement regime similarity model:
- Rolling 10 year z-score normalization (min 2 year)
- Winsorisation at +/-3
- 36 month exclusion window
- Euclidean distance metric
- Similarity weights = inverse of distance, normalised to sum to 1

In [7]:
# ==========================================================================
# Align state variables and factor returns
# ==========================================================================

common_dates = state_vars.index.intersection(factor_returns.index)
print(f'Common date range: {common_dates.min().date()} to {common_dates.max().date()}')
print(f'Common months #: {len(common_dates)}')

state_aligned = state_vars.loc[common_dates].copy()
factors_aligned = factor_returns.loc[common_dates].copy()

# Drop any state variables with > 50% missing
missing_pct = state_aligned.isna().mean()
good_vars = missing_pct[missing_pct < 0.5].index.tolist()
dropped_vars = missing_pct[missing_pct >= 0.5].index.tolist()
state_aligned = state_aligned[good_vars]

if dropped_vars:
    print(f'\n Dropped variables (>50% missing): {dropped_vars}')

print(f'\nState variables retained: {len(good_vars)}:')
for v in good_vars:
    n_valid = state_aligned[v].notna().mean()
    pct_miss = missing_pct[v] * 100
    print(f'   {v:15s}: {n_valid} obs, {pct_miss:.1f}% missing')

# Forward fill gaps of up to 3 months
print('\nForward fill missing (up to 3 months)...')
state_aligned = state_aligned.ffill(limit=3)

Common date range: 2007-07-31 to 2026-02-28
Common months #: 224

 Dropped variables (>50% missing): ['us_10y', 'real_rates']

State variables retained: 5:
   oil            : 0.9464285714285714 obs, 5.4% missing
   copper         : 0.9464285714285714 obs, 5.4% missing
   vix            : 0.9464285714285714 obs, 5.4% missing
   credit_spread  : 0.9464285714285714 obs, 5.4% missing
   dispersion     : 1.0 obs, 0.0% missing

Forward fill missing (up to 3 months)...


In [11]:
# ==========================================================================
# Distance model implementation
# ==========================================================================
# Steps:
#  1. Rolling z-score normalisation of each state variable
#  2. Winsorise at +/-3
#  3. For each month t, compute Euclidean distance to all historical months
#  4. Exclude the most recent `exclusion_window` months
#  5. Convert distances to similarity weights (inverse distance, normalised)

def fit_distance_model(state_df, norm_window=120, min_periods=24, winsorize_limit=3.0, exclusion_window=36):
    # 1. Rolling z-score
    if np.isinf(norm_window):
        rolling_mean = state_df.expanding(min_periods=min_periods).mean()
        rolling_std = state_df.expanding(min_periods=min_periods).std()
    else:
        rolling_mean = state_df.rolling(norm_window, min_periods=min_periods).mean()
        rolling_std = state_df.rolling(norm_window, min_periods=min_periods).std()
    
    normalised = (state_df - rolling_mean) / rolling_std

    # 2. Winsorize
    if winsorize_limit is not None:
        normalised = normalised.clip(-winsorize_limit, winsorize_limit)

    # 3-5. Compute distance and weights
    dates = normalised.index
    n = len(dates)
    weights_matrix = pd.DataFrame(np.nan, index=dates, columns=dates)

    for i in range(min_periods, n):
        current = normalised.iloc[i].values

        # Only compare to dates before the exclusion window
        max_j = i - exclusion_window
        if max_j <= 0:
            continue

        distances = []
        valid_indices = []

        for j in range(max_j):
            historical = normalised.iloc[j].values
            # Use only variables present in both
            valid = ~(np.isnan(current) | np.isnan(historical))
            if valid.sum() < 2:
                continue
            dist = np.sqrt(np.sum((current[valid] - historical[valid]) ** 2))
            distances.append(dist)
            valid_indices.append(j)
        
        if len(distances) == 0:
            continue

        distances = np.array(distances)

        # Convert to similarity
        max_dist = np.nanmax(distances) if len(distances) > 0 else 1.0
        if max_dist > 0:
            similarities = 1 - distances / max_dist
        else:
            similarities = np.ones_like(distances) 

        # Normalise
        max_sim = similarities.max()
        if max_sim > 0:
            similarities /= max_sim
        else:
            similarities = np.ones_like(similarities) # If all similarities are zero, treat as perfect similarity

        # Square to emphasize 
        similarities = similarities ** 2

        for k, j in enumerate(valid_indices):
            weights_matrix.iloc[i, j] = similarities[k]

    n_valid = weights_matrix.notna().any(axis=1).sum()
    print(f'Distance model fitted: {n_valid} valid periods out of {n}')
    print(f'Variables: {list(state_df.columns)}')

    return weights_matrix, normalised

# Fit model
weights_matrix, normalised_state = fit_distance_model(state_aligned, **DISTANCE_CONFIG)

Distance model fitted: 152 valid periods out of 224
Variables: ['oil', 'copper', 'vix', 'credit_spread', 'dispersion']


## 4. Similarity Analysis

Examine the similarity weights to understand which historical periods are most similar to recent conditions

In [14]:
# Visualise weights for the most recent month
latest_date = weights_matrix.index[-1]
latest_weights = weights_matrix.loc[latest_date].dropna()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=latest_weights.index,
    y=latest_weights.values,
    marker_color=COLOURS[0]
))
fig.update_layout(
    title=f'Similarity Weights: {latest_date.strftime('%Y-%m')} vs All Historical Periods',
    xaxis_title='Historical date',
    yaxis_title='Similarity weight',
    height=400,
    template='plotly_white'
)
fig.show()

# Top 5 most similar periods
top_similar = latest_weights.nlargest(10)
print(f'\nTop 10 most similar historical periods to {latest_date.strftime('%Y-%m')}')
for date, weight in top_similar.items():
    print(f'   {date.strftime('%Y-%m')}: {weight:.2f}')


Top 10 most similar historical periods to 2026-02
   2017-04: 1.00
   2017-06: 0.99
   2017-07: 0.97
   2017-11: 0.93
   2010-07: 0.92
   2013-01: 0.88
   2014-05: 0.88
   2013-10: 0.87
   2016-07: 0.87
   2017-08: 0.87


## 5. Factor Timing Backtest

Two approaches
1. **Continuous weights**: Use similarity-weighted average returns to forecast, then scale by forecast variability
2. **Quintile approach**: Classify historical dates into quintiles by similarity, go long factors with positive avg return in Q1, short if negative

In [16]:
# ==========================================================================
# Continuous weights
# ==========================================================================
# For each month t, forecast each factor's return as the similarity-weighted
# average of historical factor return. Then scale by rolling forecast std dev.

def continuous_timing_weights(weights_matrix, factor_returns, scale_window=60):
    """
    Compute continuous timing weights from similarity weighted forecasts.

    For each month t:
    1. Forecast factor return = sum(w_j * factor_return_j) for all historical j
    2. Scale by rolling std dev of forecast
    """
    common = weights_matrix.index.intersection(factor_returns.index)
    forecasts = pd.DataFrame(np.nan, index=common, columns=factor_returns.columns)

    for t in common:
        w = weights_matrix.loc[t].dropna()
        if len(w) == 0:
            continue
        hist_dates = w.index.intersection(factor_returns.index)
        if len(hist_dates) == 0:
            continue
        w_aligned = w[hist_dates]
        w_aligned = w_aligned / w_aligned.sum()
        forecasts.loc[t] = (factor_returns.loc[hist_dates].T * w_aligned).T.sum()

    # Scale by rolling std dev
    forecast_std = forecasts.rolling(scale_window, min_periods=12).std()
    timing_weights = forecasts / forecast_std.replace(0, np.nan)

    return timing_weights, forecasts

timing_weights, forecasts = continuous_timing_weights(weights_matrix, factors_aligned)

print('Continuous timing weights:')
print(f'  Shape: {timing_weights.shape}')
valid_tw = timing_weights.dropna(how='all')
print(f'  First valid: {valid_tw.index.min().date()}')
print(f'  Last valid: {valid_tw.index.max().date()}')
print(f'\nLatest weights:')
latest_w = timing_weights.iloc[-1].dropna()
print(latest_w.sort_values(ascending=False).to_string())

Continuous timing weights:
  Shape: (224, 6)
  First valid: 2014-06-30
  Last valid: 2026-02-28

Latest weights:
Market        6.0671
Momentum      3.8112
Quality       2.8304
Investment    0.9439
Size          0.0799
Value        -2.0614


In [22]:
# ==========================================================================
# Quintile
# ==========================================================================
# For each month t, classify historical dates into quintiles by similarity.
# Q1 = most similar 20%. Average factor returns in Q1.
# Long if Q1 avg > 0, short if Q1 avg < 0

def quintile_backtest(weights_matrix, factor_returns, quintile_pct=0.2, dynamic_weight=False):
    """
    Quintile-based timing backtest

    For each date t:
    1. Get similarity weights to all historical dates (excluding recent via exclusion window)
    2. Take top quintile_pct most similar dates
    3. Average factor returns in those dates
    4. Go long factor if avg > 0, short if avg < 0
    5. Portfolio return = sum of (signal * next-month factor return) / n_factors

    Returns forecast signals and portfolio returns.
    """
    dates = weights_matrix.index
    common = dates.intersection(factor_returns.index)

    signals = pd.DataFrame(np.nan, index=common, columns=factor_returns.columns)
    q1_avg_returns = pd.DataFrame(np.nan, index=common, columns=factor_returns.columns)

    for t in common:
        w = weights_matrix.loc[t].dropna()
        if len(w) < 10:
            continue

        # Top quintile by similarity weight
        threshold = w.quantile(1 - quintile_pct)
        q1_dates = w[w > threshold].index
        q1_dates = q1_dates.intersection(factor_returns.index)

        if len(q1_dates) < 5:
            continue

        # Average factor returns in Q1 dates
        q1_returns = factor_returns.loc[q1_dates].mean()
        q1_avg_returns.loc[t] = q1_returns

        # Signal
        if dynamic_weight:
            weights = q1_returns / q1_returns.abs().sum()
            signals.loc[t] = weights
        else:
            signals.loc[t] = np.sign(q1_returns)

    return signals, q1_avg_returns

signals, q1_avg = quintile_backtest(weights_matrix, factors_aligned, quintile_pct=0.2, dynamic_weight=False)
signals_dy, q1_avg_dy = quintile_backtest(weights_matrix, factors_aligned, quintile_pct=0.2, dynamic_weight=True)

print('Quintile signals:')
print(f'   Valid months: {signals.dropna(how='all').shape[0]}')
print(f'\nSignal summary (latest month)')
latest_sig = signals.iloc[-1].dropna()
print(f'\nBinary:')
print(f'{latest_sig.to_string()}')
latest_sig_dy = signals_dy.iloc[-1].dropna()
print(f'\nDynamic:')
print(f'{latest_sig_dy.to_string()}')

Quintile signals:
   Valid months: 131

Signal summary (latest month)

Binary:
Market        1.0000
Size         -1.0000
Value        -1.0000
Quality       1.0000
Investment   -1.0000
Momentum      1.0000

Dynamic:
Market        0.5234
Size         -0.0504
Value        -0.1679
Quality       0.1276
Investment   -0.0763
Momentum      0.0545


In [23]:
# ==========================================================================
# Calulate portfolio returns for both
# ==========================================================================

# Continuous weights
cont_portfolio = (timing_weights.shift(1) * factors_aligned).sum(axis=1)
cont_portfolio = cont_portfolio.dropna()
# cont_portfolio = cont_portfolio[cont_portfolio != 0]

# Quintile binary
n_factors = signals.notna().sum(axis=1)
denom = n_factors.shift(1).replace(0, np.nan)
quint_portfolio = (signals.shift(1) * factors_aligned).sum(axis=1) / denom
quint_portfolio = quint_portfolio.dropna()
# quint_portfolio = quint_portfolio[quint_portfolio != 0]

# Static long only: equal weight all (no timing)
static_portfolio = factors_aligned.mean(axis=1)

# Performance metrics
def calc_perf(returns, name):
    if len(returns) == 0:
        return {
            'Strategy': name,
            'Ann. Return (%)': np.nan,
            'Ann. Vol (%)': np.nan,
            'Sharpe': np.nan,
            'Hit Rate': np.nan,
            'Max DD (%)': np.nan,
            'Months': 0
        }
    
    ann_ret = returns.mean() * 12
    ann_vol = returns.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    cum = returns.cumsum()
    max_dd = (cum - cum.cummax()).min()
    hit_rate = (returns > 0).mean()

    return {
        'Strategy': name,
        'Ann. Return (%)': ann_ret,
        'Ann. Vol (%)': ann_vol,
        'Sharpe': sharpe,
        'Hit Rate': hit_rate,
        'Max DD (%)': max_dd,
        'Months': len(returns)
    }

# Align to common start date
starts = [s.index.min() for s in [cont_portfolio, quint_portfolio] if len(s) > 0]
if starts:
    common_start = max(starts)
else:
    common_start = static_portfolio.index.min()

perf_table = pd.DataFrame([
    calc_perf(static_portfolio[common_start:], 'Static long only'),
    calc_perf(cont_portfolio[common_start:], 'Continuous Timing'),
    calc_perf(quint_portfolio[common_start:], 'Quintile Timing')
])

print('Portfolio Performance:')
print(perf_table.to_string(index=False))

Portfolio Performance:
         Strategy  Ann. Return (%)  Ann. Vol (%)  Sharpe  Hit Rate  Max DD (%)  Months
 Static long only           0.0257        0.0447  0.5747    0.5923     -0.1063     130
Continuous Timing           0.2611        0.5097  0.5122    0.6154     -1.0243     130
  Quintile Timing          -0.0096        0.0540 -0.1786    0.5308     -0.2139     130


In [24]:
# Cumulative returns comparison
fig = go.Figure()

start = common_start

for returns, name, color in [
    (static_portfolio[start:], 'Static long only', COLOURS[0]),
    (cont_portfolio[start:], 'Coninuous Timing', COLOURS[1]),
    (quint_portfolio[start:], 'Quintile Timing', COLOURS[2]),
]:
    if len(returns) == 0:
        continue

    cum = (1 + returns).cumprod() - 1
    fig.add_trace(go.Scatter(
        x=cum.index, y=cum.values * 100,
        name=name, line=dict(color=color, width=2)
    ))

fig.update_layout(
    title='Distance Model Factor Timing: Cumulative Returns',
    xaxis_title='Date',
    yaxis_title='Cumulative Return (%)',
    height=500,
    template='plotly_white',
    legend=dict(font=dict(size=10))
)
fig.show()

## 6. Per Factor Timing Analysis

Examine which individual factors benefit most from the timing signal

In [26]:
# Per factor: compare static vs timed Sharpe ratios
timed_returns = timing_weights.shift(1) * factors_aligned

factor_perf = []
for factor in factors_aligned.columns:
    static = factors_aligned[factor].dropna()
    timed = timed_returns[factor].dropna()

    # Align
    common_idx = static.index.intersection(timed.index)
    static = static[common_idx]
    timed = timed[common_idx]

    static_sharpe = (static.mean() / static.std()) * np.sqrt(12) if static.std() > 0 else 0
    timed_sharpe = (timed.mean() / timed.std()) * np.sqrt(12) if timed.std() > 0 else 0

    factor_perf.append({
        'Factor': factor,
        'Static Sharpe': static_sharpe,
        'Timed Sharpe': timed_sharpe,
        'Improvement': round(timed_sharpe - static_sharpe, 3),
        'Hit Rate': round((timed > 0).mean(), 3),
        'Months': len(common_idx)
    })

factor_perf_df = pd.DataFrame(factor_perf).sort_values('Improvement', ascending=False)
print('Per factor timing performance (Continuous):')
print(factor_perf_df.to_string(index=False))

Per factor timing performance (Continuous):
    Factor  Static Sharpe  Timed Sharpe  Improvement  Hit Rate  Months
      Size        -0.2301        0.4090       0.6390    0.5290     140
   Quality         0.4855        0.6074       0.1220    0.5790     140
Investment        -0.0812       -0.1123      -0.0310    0.4930     140
     Value        -0.0918       -0.1437      -0.0520    0.5790     140
  Momentum         0.2001        0.1341      -0.0660    0.5430     140
    Market         0.7666        0.6106      -0.1560    0.6430     140


In [28]:
# Chart: Static vs Timed Sharpe by Factor
fig = go.Figure()

factors_sorted = factor_perf_df.sort_values('Factor', ascending=False)

fig.add_trace(go.Bar(
    y=factors_sorted['Factor'],
    x=factors_sorted['Static Sharpe'],
    name='Static (No Timing)',
    orientation='h',
    marker_color=COLOURS[0]
))
fig.add_trace(go.Bar(
    y=factors_sorted['Factor'],
    x=factors_sorted['Timed Sharpe'],
    name='Distance Model Timing',
    orientation='h',
    marker_color=COLOURS[1]
))

fig.update_layout(
    title='Factor Sharpe Ratio: Static vs Distance Model Timing',
    xaxis_title='Annualised Sharpe Ratio',
    barmode='group',
    height=450,
    template='plotly_white',
    legend=dict(font=dict(size=10))
)
fig.show()

## 7. State Variable Cross Correlations

Verify that the state variables provide independent information

In [33]:
# State variable correlation matrix
state_corr = state_aligned.corr()

fig = ff.create_annotated_heatmap(
    z=state_corr.values.round(2),
    x=[str(c)[:20] for c in state_corr.columns],
    y=[str(c)[:20] for c in state_corr.index],
    colorscale='RdBu_r',
    zmin=-1, zmax=1,
    showscale=True,
    font_colors=['black']
)
fig.update_layout(
    title='State Variable Cross Correlations (12M changes)',
    height=max(400, len(state_corr) * 50),
    width=max(500, len(state_corr) * 50),
    font=dict(size=9)
)
fig.show()

# Average absolute correlation
upper_tri = state_corr.where(np.triu(np.ones(state_corr.shape), k=1).astype(bool))
avg_abs = upper_tri.abs().stack().mean()
print(f'\nAverage absolute cross correlation: {avg_abs:.2f}')


Average absolute cross correlation: 0.45
